# Advanced Firestore Queries


Welcome back! In previous lessons, you learned how to retrieve individual documents and multiple documents from Firestore collections. Today, we'll explore more advanced ways to retrieve data by using queries and filters.

These techniques allow you to efficiently find documents that match specific criteria, making it easier to work with large and complex datasets. For our examples, we'll use a `books` collection, where each document represents a book with fields such as `year`, `title`, and `author`.

---

## Firestore Querying Model Overview

Firestore provides a powerful querying model that lets you filter documents within a collection based on field values. You can retrieve all documents that match one or more conditions, such as finding all books published in a certain year or by a specific author. Queries in Firestore are always performed within a single collection and can use a variety of comparison operators.

Unlike some databases, Firestore does not have a "scan" operation that reads every document in a collection. Instead, it uses pre-computed indexes to retrieve only the documents that match your specified conditions, which ensures performance remains efficient and scalable regardless of your dataset size.

---

## Crafting and Enhancing Queries

To create queries in Firestore, you use method chaining to add filters and select specific fields. Here are some common ways to build and execute queries:

### Basic Query: Filtering by a Single Field

To retrieve all books published in a specific year, use the `where()` method combined with a `FieldFilter`:

```python
from google.cloud import firestore
from google.cloud.firestore import FieldFilter

db = firestore.Client()
books_ref = db.collection('books')

# Define and stream the query
query = books_ref.where(filter=FieldFilter('year', '==', 2018))
results = query.stream()

for doc in results:
    print(doc.id, doc.to_dict())

```

**Example Output:**

> `book_001 {'title': 'The Silent Patient', 'author': 'Alex Michaelides', 'year': 2018, 'genre': 'thriller'}`
> `book_003 {'title': 'Educated', 'author': 'Tara Westover', 'year': 2018, 'genre': 'memoir'}`
> `book_007 {'title': 'Becoming', 'author': 'Michelle Obama', 'year': 2018, 'genre': 'autobiography'}`

### Querying with Multiple Conditions

You can chain multiple `where()` calls to filter documents by more than one field. For example, to find all books published in 2015 with a title that starts with "The":

```python
# Range query on strings to simulate a "begins with" filter
query = books_ref.where(filter=FieldFilter('year', '==', 2015)) \
                 .where(filter=FieldFilter('title', '>=', 'The')) \
                 .where(filter=FieldFilter('title', '<', 'Thf'))

results = query.stream()
for doc in results:
    print(doc.id, doc.to_dict())

```

**Example Output:**

> `book_012 {'title': 'The Girl on the Train', 'author': 'Paula Hawkins', 'year': 2015, 'genre': 'thriller'}`
> `book_018 {'title': 'The Martian', 'author': 'Andy Weir', 'year': 2015, 'genre': 'science fiction'}`

> 📌 **Note:** Firestore does not support "begins with" queries directly, but you can use range queries on strings (incrementing the last character) to approximate this behavior.

### Selecting Specific Fields (Projection)

To minimize data transfer overhead across the network, use the `select()` method to return only the fields your application needs:

```python
query = books_ref.where(filter=FieldFilter('year', '==', 2015)).select(['year', 'title', 'author'])
results = query.stream()

for doc in results:
    print(doc.id, doc.to_dict())

```

**Example Output:**

> `book_012 {'year': 2015, 'title': 'The Girl on the Train', 'author': 'Paula Hawkins'}`
> `book_015 {'year': 2015, 'title': 'Go Set a Watchman', 'author': 'Harper Lee'}`
> `book_018 {'year': 2015, 'title': 'The Martian', 'author': 'Andy Weir'}`

---

## Using Comparison Operators

Firestore supports several comparison operators to evaluate document properties:

| Operator | Description |
| --- | --- |
| `==` | Equal to |
| `!=` | Not equal to |
| `<`, `<=`, `>`, `>=` | Less than, less than or equal to, greater than, greater than or equal to |
| `in` | Field matches any value in a provided list (up to 10 values) |
| `array_contains` | Array field contains a specific scalar value |

### Example: Using the `in` Operator

To find all books published in either 2017 or 2018:

```python
query = books_ref.where(filter=FieldFilter('year', 'in', [2017, 2018]))
results = query.stream()

for doc in results:
    print(doc.id, doc.to_dict())

```

**Example Output:**

> `book_001 {'title': 'The Silent Patient', 'author': 'Alex Michaelides', 'year': 2018, 'genre': 'thriller'}`
> `book_003 {'title': 'Educated', 'author': 'Tara Westover', 'year': 2018, 'genre': 'memoir'}`
> `book_005 {'title': "The Handmaid's Tale", 'author': 'Margaret Atwood', 'year': 2017, 'genre': 'dystopian'}`

---

## Limitations and Considerations

* **Query Result Size:** A single query stream chunk can safely handle data scaling, but large datasets should utilize pagination to optimize client-side memory performance.
* **Index Requirements:** Simple single-field queries are indexed automatically. However, compound queries (filtering across multiple fields with inequalities or sorting) require a composite index. If an index is missing, the Firestore SDK raises an exception containing a direct URL link to generate it in the Google Cloud Console.
* **Single Collection Scope:** Standard queries run within the boundary of a single collection. You cannot perform SQL-like relational `JOIN` operations across different collections natively.
* **Inequality Filters:** If you include inequality filters (`!=`, `<`, `>`, `>=`), any other field filters containing inequalities must target that exact same field.
* **No Collection Scans:** Firestore enforces strict query planning; it does not support brute-force database scans without defined constraints, ensuring execution speeds stay lightning-fast.

---

## Summary and Looking Ahead

In this lesson, you mastered Firestore's core querying engine:

* Filtering documents with single and chained compound `FieldFilter` conditions.
* Optimizing network bandwidth with field projection using `.select()`.
* Evaluating datasets with operators like `==`, inequalities, and membership checks (`in`).

Practice designing various query combinations to master data indexing behaviors. This foundation prepares you for advanced schema manipulation and high-performance querying architectures!

## Introduction to Querying in Firestore

This task is designed to familiarize you with executing Firestore Query operations using a provided Python script. You will run the script that interacts with a pre-configured Movies collection, which includes operations to retrieve data using various Query configurations. Observe how data is retrieved under different scenarios, including using simple conditions, field selection, and multiple filter expressions. This exercise will help enhance your understanding of data retrieval dynamics in Firestore.

Important Note: Running scripts can modify resources in our GCP simulator. To revert to the initial state, use the reset button located in the top right corner. Resetting will erase any code changes, so copy your code to the clipboard to preserve it during a reset.

```python
from google.cloud import firestore
from google.cloud.firestore import FieldFilter

# Initialize Firestore client
db = firestore.Client()

# Create reference to Movies collection
movies_collection = db.collection('Movies')

# Populate collection with data
movies_collection.add({'year': 2016, 'title': 'The Big New Movie'})
movies_collection.add({'year': 2017, 'title': 'The Bigger, Newer Movie'})
movies_collection.add({'year': 2017, 'title': 'Yet Another Movie'})
movies_collection.add({'year': 2017, 'title': 'One More Movie'})
movies_collection.add({'year': 2015, 'title': 'An Old Movie'})
movies_collection.add({'year': 2018, 'title': 'Another New Movie'})

# Example with Query command 1: Simple Query
query_1 = movies_collection.where(filter=FieldFilter('year', '==', 2018))
query_1_results = [doc.to_dict() for doc in query_1.stream()]
print("Query 1 Results:", query_1_results)

# Example with Query command 2: Query with field selection
query_2 = movies_collection.where(filter=FieldFilter('year', '==', 2015)).select(['year', 'title'])
query_2_results = [doc.to_dict() for doc in query_2.stream()]
print("Query 2 Results:", query_2_results)

# Example with Query command 3: Retrieve all documents from collection
query_3 = movies_collection
query_3_results = [doc.to_dict() for doc in query_3.stream()]
print("Query 3 Results (All Documents):", query_3_results)

# Example with Query command 4: Multiple condition query
query_4 = movies_collection.where(filter=FieldFilter('year', 'in', [2017, 2018]))
query_4_results = [doc.to_dict() for doc in query_4.stream()]
print("Query 4 Results (Multiple Years):", query_4_results)

```

Running this script highlights the fundamental ways to extract data from a NoSQL database. Rather than searching through every single row like a traditional relational scan, Firestore relies entirely on backend indexes, allowing these queries to return data instantly regardless of the size of your collection.

Here is an architectural breakdown of what happens behind the scenes for each query configuration.

---

## 1. Document Creation Twist: `add()` vs `set()`

Unlike previous scripts where you used `.document('1').set(...)`, this script uses `movies_collection.add(...)`.

* When you call `.add()`, you don't provide a document ID.
* Firestore automatically assigns a randomized, unique 20-character alphanumeric hash string (e.g., `qW3b7RxlPZ91KjM...`) to act as the key field.

---

## 2. Breaking Down the Query Configurations

### Query 1: The Single Equality Filter

```python
query_1 = movies_collection.where(filter=FieldFilter('year', '==', 2018))

```

* **Mechanics:** Firestore immediately looks up the pre-built internal index for the `year` field. It jumps directly to the entry for `2018` and fetches the corresponding document snapshots.
* **Expected Output:** Returns a list containing one dictionary: `Another New Movie`.

### Query 2: Field Selection (Projection Optimization)

```python
query_2 = movies_collection.where(filter=FieldFilter('year', '==', 2015)).select(['year', 'title'])

```

* **Mechanics:** While this looks similar to Query 1, the trailing `.select([...])` alters the server pipeline. The Firestore server extracts the document, strips out any non-specified fields (such as `genre` if it existed), and delivers a lighter payload over the network.
* **Expected Output:** Returns the data for `An Old Movie`, explicitly showing only the specified attributes.

### Query 3: Streaming the Entire Collection

```python
query_3 = movies_collection
query_3_results = [doc.to_dict() for doc in query_3.stream()]

```

* **Mechanics:** Calling `.stream()` directly on a collection reference retrieves all records currently present in that segment. Firestore pulls these documents via a streaming RPC call, allowing your client app to handle entries continuously as they arrive over the wire.
* **Expected Output:** A collection of all 6 generated movie documents.

### Query 4: Set Membership with the `in` Operator

```python
query_4 = movies_collection.where(filter=FieldFilter('year', 'in', [2017, 2018]))

```

* **Mechanics:** The `in` operator lets you pass a list of up to 10 values. Under the hood, Firestore splits this into parallel sub-queries (`year == 2017` and `year == 2018`) and merges the resulting streams seamlessly before handing them back to your code.
* **Expected Output:** Returns 4 records: the three movies from 2017 alongside the single movie from 2018.

---

## Summary of Query Behaviors

| Feature | Method | Network Payload | Best For... |
| --- | --- | --- | --- |
| **Direct Filter** | `.where(filter=...)` | Full Document Map | Basic target searches. |
| **Field Projection** | `.select([...])` | Targeted Key/Value Only | Mobile or web views needing small data chunks. |
| **Collection Stream** | `.stream()` on Collection | Entire Collection | Backups, reporting, or localized cache syncs. |
| **Set Containment** | `'in'` operator | Combined Query Matches | Matching against a specific group of IDs or tags. |

## Firestore Compound Queries with Multiple Conditions

CosmoYesterday

This task is designed to familiarize you with executing Firestore Query operations using a provided Python script. You will run the script that interacts with a pre-configured Movies collection, which includes operations to retrieve data using various Query configurations. Observe how data is retrieved under different scenarios, including using simple conditions, field selection, and multiple filter expressions. This exercise will help enhance your understanding of data retrieval dynamics in Firestore.

Important Note: Running scripts can modify resources in our GCP simulator. To revert to the initial state, use the reset button located in the top right corner. Resetting will erase any code changes, so copy your code to the clipboard to preserve it during a reset.

```python
from google.cloud import firestore
from google.cloud.firestore import FieldFilter

# Initialize Firestore client
db = firestore.Client()

# Create reference to Movies collection
movies_collection = db.collection('Movies')

# Populate collection with data
movies_collection.add({'year': 2016, 'title': 'The Big New Movie'})
movies_collection.add({'year': 2017, 'title': 'The Bigger, Newer Movie'})
movies_collection.add({'year': 2017, 'title': 'Yet Another Movie'})
movies_collection.add({'year': 2017, 'title': 'One More Movie'})
movies_collection.add({'year': 2015, 'title': 'An Old Movie'})
movies_collection.add({'year': 2018, 'title': 'Another New Movie'})

# Example with Query command 1: Simple Query
query_1 = movies_collection.where(filter=FieldFilter('year', '==', 2018))
query_1_results = [doc.to_dict() for doc in query_1.stream()]
print("Query 1 Results:", query_1_results)

# Example with Query command 2: Query with field selection
query_2 = movies_collection.where(filter=FieldFilter('year', '==', 2015)).select(['year', 'title'])
query_2_results = [doc.to_dict() for doc in query_2.stream()]
print("Query 2 Results:", query_2_results)

# Example with Query command 3: Retrieve all documents from collection
query_3 = movies_collection
query_3_results = [doc.to_dict() for doc in query_3.stream()]
print("Query 3 Results (All Documents):", query_3_results)

# Example with Query command 4: Multiple condition query
query_4 = movies_collection.where(filter=FieldFilter('year', 'in', [2017, 2018]))
query_4_results = [doc.to_dict() for doc in query_4.stream()]
print("Query 4 Results (Multiple Years):", query_4_results)

```

Here is the complete, fully functional Python script containing all the Firestore querying strategies we have discussed. It includes data population using auto-generated IDs, simple filtering, field projection, full collection streaming, set membership matching, and an advanced compound multi-condition query.

```python
from google.cloud import firestore
from google.cloud.firestore import FieldFilter

# ==========================================
# 1. INITIALIZE CLIENT & POPULATE DATA
# ==========================================
db = firestore.Client()
movies_collection = db.collection('Movies')

print("Populating Movies collection...")
# Using .add() to let Firestore auto-generate unique document IDs
movies_collection.add({'year': 2016, 'title': 'The Big New Movie', 'genre': 'Action'})
movies_collection.add({'year': 2017, 'title': 'The Bigger, Newer Movie', 'genre': 'Comedy'})
movies_collection.add({'year': 2017, 'title': 'Yet Another Movie', 'genre': 'Drama'})
movies_collection.add({'year': 2017, 'title': 'One More Movie', 'genre': 'Action'})
movies_collection.add({'year': 2015, 'title': 'An Old Movie', 'genre': 'Classic'})
movies_collection.add({'year': 2018, 'title': 'Another New Movie', 'genre': 'Sci-Fi'})
print("Data population complete.\n")

# ==========================================
# 2. QUERY 1: SIMPLE EQUALITY FILTER
# ==========================================
print("--- Query 1: Simple Filter (Year == 2018) ---")
query_1 = movies_collection.where(filter=FieldFilter('year', '==', 2018))
query_1_results = [doc.to_dict() for doc in query_1.stream()]
print(f"Results: {query_1_results}\n")

# ==========================================
# 3. QUERY 2: FIELD PROJECTION (OPTIMIZED PAYLOAD)
# ==========================================
print("--- Query 2: Field Selection (Title & Year Only for 2015) ---")
query_2 = movies_collection.where(filter=FieldFilter('year', '==', 2015)).select(['year', 'title'])
query_2_results = [doc.to_dict() for doc in query_2.stream()]
print(f"Results (Genre omitted): {query_2_results}\n")

# ==========================================
# 4. QUERY 3: FULL COLLECTION STREAM (SCAN SIMULATION)
# ==========================================
print("--- Query 3: Stream All Documents in Collection ---")
query_3_results = []
for doc in movies_collection.stream():
    query_3_results.append({'doc_id': doc.id, 'data': doc.to_dict()})
print(f"Total documents retrieved: {len(query_3_results)}")
print(f"Sample Document Sample: {query_3_results[0]}\n")

# ==========================================
# 5. QUERY 4: SET MEMBERSHIP MATCHING ('in' OPERATOR)
# ==========================================
print("--- Query 4: Set Membership (Year in [2017, 2018]) ---")
query_4 = movies_collection.where(filter=FieldFilter('year', 'in', [2017, 2018]))
query_4_results = [doc.to_dict() for doc in query_4.stream()]
print(f"Found {len(query_4_results)} movies from 2017/2018: {query_4_results}\n")

# ==========================================
# 6. QUERY 5: ADVANCED COMPOUND MULTI-CONDITION QUERY
# ==========================================
print("--- Query 5: Compound Query (Year == 2017 AND Genre == 'Action') ---")
# Chaining multiple .where() filters defines an AND relationship
query_5 = movies_collection.where(filter=FieldFilter('year', '==', 2017)) \
                           .where(filter=FieldFilter('genre', '==', 'Action'))

query_5_results = [doc.to_dict() for doc in query_5.stream()]
print(f"Results matching both conditions: {query_5_results}\n")

```

---

### Execution Checklist

* **Authentication**: Ensure your `GOOGLE_APPLICATION_CREDENTIALS` environment variable is set or you are running this within an authenticated Google Cloud Cloud Shell session.
* **Composite Index Warning**: Query 5 evaluates filters across two distinct fields (`year` and `genre`). If you run this exact compound structure in a brand new Google Cloud project, Firestore might initially throw an `FailedPrecondition` exception. The error message will contain a direct, secure hyperlink—clicking it instantly logs into your GCP console to automatically build the required composite index.

## Executing Firestore Query and Scan Operations


Great progress on your Firestore journey! This exercise introduces you to compound queries, a means to construct multiple conditions for a Firestore query operation. You will use Python script that initializes a Firestore collection Movies and populates it with movie records.

Your task involves modifying an existing query. Initially, it fetches movies from a specific year. Adjust it to retrieve only the movies from 2016 with titles starting with "The". This modification will showcase the power and flexibility of query conditions in narrowing down search results.

Important Note: Running scripts can lead to alterations in our GCP simulator resources. To return to the starting state, use the reset button in the top right corner. Be aware, resetting will erase any code changes. To safeguard your code during a reset, consider copying it to the clipboard.

```python
from google.cloud import firestore
from google.cloud.firestore import FieldFilter

# Initialize Firestore client
db = firestore.Client()

# Reference to the Movies collection
movies_ref = db.collection('Movies')

# Clear any existing data in the collection
docs = movies_ref.stream()
for doc in docs:
    doc.reference.delete()

# Fill the collection with data
movies_ref.add({'year': 2016, 'title': 'The Big New Movie'})
movies_ref.add({'year': 2016, 'title': 'The Bigger, Newer Movie'})
movies_ref.add({'year': 2016, 'title': 'Yet Another Movie'})
movies_ref.add({'year': 2018, 'title': 'One More Movie'})

# --- TODO FIXED: Compound Query for Year 2016 and Titles starting with 'The' ---
query = movies_ref.where(filter=FieldFilter('year', '==', 2016)) \
                  .where(filter=FieldFilter('title', '>=', 'The')) \
                  .where(filter=FieldFilter('title', '<', 'Thf'))

results = query.stream()

query_results = []
for doc in results:
    data = doc.to_dict()
    query_results.append(data)

print("Query Results:", query_results)

```

Here is the complete, fully functional Python script containing all the Firestore querying strategies we have discussed. It includes data population using auto-generated IDs, simple filtering, field projection, full collection streaming, set membership matching, and an advanced compound multi-condition query.

```python
from google.cloud import firestore
from google.cloud.firestore import FieldFilter

# ==========================================
# 1. INITIALIZE CLIENT & POPULATE DATA
# ==========================================
db = firestore.Client()
movies_collection = db.collection('Movies')

print("Populating Movies collection...")
# Using .add() to let Firestore auto-generate unique document IDs
movies_collection.add({'year': 2016, 'title': 'The Big New Movie', 'genre': 'Action'})
movies_collection.add({'year': 2017, 'title': 'The Bigger, Newer Movie', 'genre': 'Comedy'})
movies_collection.add({'year': 2017, 'title': 'Yet Another Movie', 'genre': 'Drama'})
movies_collection.add({'year': 2017, 'title': 'One More Movie', 'genre': 'Action'})
movies_collection.add({'year': 2015, 'title': 'An Old Movie', 'genre': 'Classic'})
movies_collection.add({'year': 2018, 'title': 'Another New Movie', 'genre': 'Sci-Fi'})
print("Data population complete.\n")

# ==========================================
# 2. QUERY 1: SIMPLE EQUALITY FILTER
# ==========================================
print("--- Query 1: Simple Filter (Year == 2018) ---")
query_1 = movies_collection.where(filter=FieldFilter('year', '==', 2018))
query_1_results = [doc.to_dict() for doc in query_1.stream()]
print(f"Results: {query_1_results}\n")

# ==========================================
# 3. QUERY 2: FIELD PROJECTION (OPTIMIZED PAYLOAD)
# ==========================================
print("--- Query 2: Field Selection (Title & Year Only for 2015) ---")
query_2 = movies_collection.where(filter=FieldFilter('year', '==', 2015)).select(['year', 'title'])
query_2_results = [doc.to_dict() for doc in query_2.stream()]
print(f"Results (Genre omitted): {query_2_results}\n")

# ==========================================
# 4. QUERY 3: FULL COLLECTION STREAM (SCAN SIMULATION)
# ==========================================
print("--- Query 3: Stream All Documents in Collection ---")
query_3_results = []
for doc in movies_collection.stream():
    query_3_results.append({'doc_id': doc.id, 'data': doc.to_dict()})
print(f"Total documents retrieved: {len(query_3_results)}")
print(f"Sample Document Sample: {query_3_results[0]}\n")

# ==========================================
# 5. QUERY 4: SET MEMBERSHIP MATCHING ('in' OPERATOR)
# ==========================================
print("--- Query 4: Set Membership (Year in [2017, 2018]) ---")
query_4 = movies_collection.where(filter=FieldFilter('year', 'in', [2017, 2018]))
query_4_results = [doc.to_dict() for doc in query_4.stream()]
print(f"Found {len(query_4_results)} movies from 2017/2018: {query_4_results}\n")

# ==========================================
# 6. QUERY 5: ADVANCED COMPOUND MULTI-CONDITION QUERY
# ==========================================
print("--- Query 5: Compound Query (Year == 2017 AND Genre == 'Action') ---")
# Chaining multiple .where() filters defines an AND relationship
query_5 = movies_collection.where(filter=FieldFilter('year', '==', 2017)) \
                           .where(filter=FieldFilter('genre', '==', 'Action'))

query_5_results = [doc.to_dict() for doc in query_5.stream()]
print(f"Results matching both conditions: {query_5_results}\n")

```

---

### Execution Checklist

* **Authentication**: Ensure your `GOOGLE_APPLICATION_CREDENTIALS` environment variable is set or you are running this within an authenticated Google Cloud Cloud Shell session.
* **Composite Index Warning**: Query 5 evaluates filters across two distinct fields (`year` and `genre`). If you run this exact compound structure in a brand new Google Cloud project, Firestore might initially throw an `FailedPrecondition` exception. The error message will contain a direct, secure hyperlink—clicking it instantly logs into your GCP console to automatically build the required composite index.

## Firestore Compound Where Queries with Multiple Filter Conditions

In this exercise, you'll deepen your understanding of Firestore's where queries, which can filter documents in a collection based on specific criteria. This operation can be particularly useful when you need to examine a broad dataset with multiple filtering conditions. Your task is to extend the existing Python script from the previous exercise that created a Firestore collection named Movies. You'll now add a genre attribute to the existing movie records and add additional movies to demonstrate filtering. Then you'll modify the where queries in the script to filter for movies in the Action genre that were filmed after 2017. This will demonstrate how to refine where queries to retrieve specific data efficiently while building upon your previous filtering knowledge.

Important Note: Running scripts can modify the resources in our GCP simulator. To revert to the initial state, you can use the reset button located in the top right corner. However, keep in mind that resetting will erase any code changes. To preserve your code during a reset, consider copying it to the clipboard.

```python
from google.cloud import firestore
from google.cloud.firestore import FieldFilter

# Initialize Firestore client
db = firestore.Client()

# Reference to the Movies collection
movies_ref = db.collection('Movies')

# Clear any existing data in the collection
docs = movies_ref.stream()
for doc in docs:
    doc.reference.delete()

# Start with the previous task's movies and add genre attribute
movies_ref.add({'year': 2016, 'title': 'The Big New Movie', 'genre': 'Comedy'})
movies_ref.add({'year': 2016, 'title': 'The Bigger, Newer Movie', 'genre': 'Action'})
movies_ref.add({'year': 2016, 'title': 'Yet Another Movie', 'genre': 'Drama'})
movies_ref.add({'year': 2018, 'title': 'One More Movie', 'genre': 'Action'})

# Add additional movies to demonstrate Action genre filtering after 2017
movies_ref.add({'year': 2017, 'title': 'Action Hero Returns', 'genre': 'Action'})
movies_ref.add({'year': 2015, 'title': 'An Old Movie', 'genre': 'Romance'})
movies_ref.add({'year': 2018, 'title': 'Another New Movie', 'genre': 'Action'})
movies_ref.add({'year': 2019, 'title': 'Latest Action Flick', 'genre': 'Action'})

# First show the previous task's query (movies from 2016 beginning with 'The')
print("=== Previous Task Query Results ===")
previous_query = movies_ref.where(filter=FieldFilter('year', '==', 2016)).where(filter=FieldFilter('title', '>=', 'The')).where(filter=FieldFilter('title', '<', 'Thf'))
previous_results = previous_query.stream()

previous_query_results = []
for doc in previous_results:
    data = doc.to_dict()
    previous_query_results.append(data)

print("Movies from 2016 beginning with 'The':", previous_query_results)

# TODO: Modify the query to filter for Action movies filmed after 2017
print("\n=== Current Task Query Results ===")
query = movies_ref.where(filter=FieldFilter('genre', '==', 'Drama'))
results = query.stream()

query_results = []
for doc in results:
    data = doc.to_dict()
    query_results.append(data)

print("Query Results for 'Drama' Movies:", query_results)
```

To achieve this filtering, you need to combine an **equality filter** (`==` for the genre) with an **inequality filter** (`>` for the year). Chaining these methods together creates an implicit logical `AND` condition.

Here is the complete, updated script ready to be executed:

```python
from google.cloud import firestore
from google.cloud.firestore import FieldFilter

# Initialize Firestore client
db = firestore.Client()

# Reference to the Movies collection
movies_ref = db.collection('Movies')

# Clear any existing data in the collection
docs = movies_ref.stream()
for doc in docs:
    doc.reference.delete()

# Start with the previous task's movies and add genre attribute
movies_ref.add({'year': 2016, 'title': 'The Big New Movie', 'genre': 'Comedy'})
movies_ref.add({'year': 2016, 'title': 'The Bigger, Newer Movie', 'genre': 'Action'})
movies_ref.add({'year': 2016, 'title': 'Yet Another Movie', 'genre': 'Drama'})
movies_ref.add({'year': 2018, 'title': 'One More Movie', 'genre': 'Action'})

# Add additional movies to demonstrate Action genre filtering after 2017
movies_ref.add({'year': 2017, 'title': 'Action Hero Returns', 'genre': 'Action'})
movies_ref.add({'year': 2015, 'title': 'An Old Movie', 'genre': 'Romance'})
movies_ref.add({'year': 2018, 'title': 'Another New Movie', 'genre': 'Action'})
movies_ref.add({'year': 2019, 'title': 'Latest Action Flick', 'genre': 'Action'})

# First show the previous task's query (movies from 2016 beginning with 'The')
print("=== Previous Task Query Results ===")
previous_query = movies_ref.where(filter=FieldFilter('year', '==', 2016)).where(filter=FieldFilter('title', '>=', 'The')).where(filter=FieldFilter('title', '<', 'Thf'))
previous_results = previous_query.stream()

previous_query_results = []
for doc in previous_results:
    data = doc.to_dict()
    previous_query_results.append(data)

print("Movies from 2016 beginning with 'The':", previous_query_results)

# --- TODO FIXED: Modify the query to filter for Action movies filmed after 2017 ---
print("\n=== Current Task Query Results ===")
query = movies_ref.where(filter=FieldFilter('genre', '==', 'Action')) \
                  .where(filter=FieldFilter('year', '>', 2017))
results = query.stream()

query_results = []
for doc in results:
    data = doc.to_dict()
    query_results.append(data)

print("Action Movies filmed after 2017:", query_results)

```

---

### Key Query Mechanics to Remember

* **Implicit AND Engine:** Chaining multiple `.where()` calls automatically tells Firestore that a document must meet *every* criterion to be included in the results.
* **The Composite Index Rule:** Because this query filters by equality on one field (`genre`) and an inequality range on a completely different field (`year`), Firestore relies on a **Composite Index**.
* **Production Exception Handling:** In a real cloud project (outside the simulator), running a compound query across different fields for the first time will trigger a `FailedPrecondition` exception. The error message will provide a direct Google Cloud Console hyperlink. Clicking it instantly auto-builds the exact composite index your query requires.

## Firestore Where Clause Query Implementation

You are progressing well on your Firestore journey! For this task, you are provided with a Python script that initializes a Firestore collection named Movies and populates it with several movie records. Each record includes a year and title attribute. Your task is to extend the script by writing two where clause queries:

    Retrieve all movies released in the year 2018.
    Retrieve all movies from 2017 that have the title starting with "The".

This exercise aims to enhance your ability to construct and execute where clause queries, a crucial skill for efficiently retrieving specific data sets from Firestore.

Important Note: Running scripts can modify the resources in our GCP simulator. To revert to the initial state, you can use the reset button located in the top right corner. However, keep in mind that resetting will erase any code changes. To preserve your code during a reset, consider copying it to the clipboard.


```python
from google.cloud import firestore

# Initialize Firestore client
db = firestore.Client()

# Reference to the Movies collection
movies_ref = db.collection('Movies')

# Clear any existing data in the collection
docs = movies_ref.stream()
for doc in docs:
    doc.reference.delete()

# Populate collection with movie data
movies_ref.add({'year': 2016, 'title': 'The Big New Movie'})
movies_ref.add({'year': 2017, 'title': 'The Bigger, Newer Movie'})
movies_ref.add({'year': 2017, 'title': 'Yet Another Movie'})
movies_ref.add({'year': 2017, 'title': 'One More Movie'})
movies_ref.add({'year': 2015, 'title': 'An Old Movie'})
movies_ref.add({'year': 2018, 'title': 'Another New Movie'})

# TODO: Write a query to retrieve all movies released in 2018 and print them

# TODO: Write a query to retrieve all movies from 2017 with titles starting with "The" and print them

```

Here is the fully completed script with both the simple query and the multi-condition compound query implemented.

To ensure the script executes correctly using modern Firestore best practices, the `FieldFilter` class has been added to the imports.

```python
from google.cloud import firestore
from google.cloud.firestore import FieldFilter

# Initialize Firestore client
db = firestore.Client()

# Reference to the Movies collection
movies_ref = db.collection('Movies')

# Clear any existing data in the collection
docs = movies_ref.stream()
for doc in docs:
    doc.reference.delete()

# Populate collection with movie data
movies_ref.add({'year': 2016, 'title': 'The Big New Movie'})
movies_ref.add({'year': 2017, 'title': 'The Bigger, Newer Movie'})
movies_ref.add({'year': 2017, 'title': 'Yet Another Movie'})
movies_ref.add({'year': 2017, 'title': 'One More Movie'})
movies_ref.add({'year': 2015, 'title': 'An Old Movie'})
movies_ref.add({'year': 2018, 'title': 'Another New Movie'})

# --- TODO FIXED: Query to retrieve all movies released in 2018 ---
print("=== Movies Released in 2018 ===")
query_2018 = movies_ref.where(filter=FieldFilter('year', '==', 2018))
results_2018 = query_2018.stream()

for doc in results_2018:
    print(f"Found: {doc.to_dict()}")

# --- TODO FIXED: Query to retrieve movies from 2017 starting with 'The' ---
print("\n=== Movies from 2017 Starting with 'The' ===")
# We chain an equality check with a string range query to simulate a 'starts with' filter
query_2017_the = movies_ref.where(filter=FieldFilter('year', '==', 2017)) \
                           .where(filter=FieldFilter('title', '>=', 'The')) \
                           .where(filter=FieldFilter('title', '<', 'Thf'))
results_2017_the = query_2017_the.stream()

for doc in results_2017_the:
    print(f"Found: {doc.to_dict()}")

```

### Pro-Tips for this Exercise

* **String Range Trick:** Because NoSQL structures lack an explicit `LIKE 'The%'` operator, enforcing a range boundary (`>= 'The'` and `< 'Thf'`) instructs the index scanner to cleanly slice out strings matching that exact prefix.
* **Chaining as an `AND` Junction:** Chaining multiple `.where()` filters establishes an implicit logical `AND` constraint. Documents must satisfy every single row boundary condition to be yielded into the client stream.

## Implementing Firestore Querying Operations for Movies

Great job progressing in your Firestore learning journey! In this task, you'll enhance your understanding of querying operations, which allow for comprehensive data retrieval across a Firestore collection. The collection, named Movies, already contains multiple movie records, each with attributes like year, title, and genre. Your challenge is to implement several querying operations ranging from simple to complex to filter movies based on different criteria. This exercise aims to showcase the versatility and broad applicability of Firestore queries compared to more specific query operations.

Important Note: Running scripts can modify the resources in our GCP simulator. To revert to the initial state, you can use the reset button located in the top right corner. However, keep in mind that resetting will erase any code changes. To preserve your code during a reset, consider copying it to the clipboard.

```python
from google.cloud import firestore
from google.cloud.firestore import FieldFilter

# Initialize Firestore client
db = firestore.Client()

# Reference to the Movies collection
movies_ref = db.collection('Movies')

# Clear any existing data in the collection
docs = movies_ref.stream()
for doc in docs:
    doc.reference.delete()

# Populate collection with movie data including the genre attribute
movies_ref.add({'year': 2016, 'title': 'The Big New Movie', 'genre': 'Comedy'})
movies_ref.add({'year': 2017, 'title': 'The Bigger, Newer Movie', 'genre': 'Action'})
movies_ref.add({'year': 2017, 'title': 'Yet Another Movie', 'genre': 'Action'})
movies_ref.add({'year': 2017, 'title': 'One More Movie', 'genre': 'Drama'})
movies_ref.add({'year': 2015, 'title': 'An Old Movie', 'genre': 'Romance'})
movies_ref.add({'year': 2018, 'title': 'Another New Movie', 'genre': 'Action'})

# Query for all movies released in 2018
query_2018 = movies_ref.where(filter=FieldFilter('year', '==', 2018))
results_2018 = query_2018.stream()

movies_2018 = []
for doc in results_2018:
    movies_2018.append(doc.to_dict())

print("Movies from 2018:", movies_2018)

# Query for all movies from 2017 with titles starting with "The"
query_2017_the = movies_ref.where(filter=FieldFilter('year', '==', 2017)).where(filter=FieldFilter('title', '>=', 'The')).where(filter=FieldFilter('title', '<', 'Thf'))
results_2017_the = query_2017_the.stream()

movies_2017_the = []
for doc in results_2017_the:
    movies_2017_the.append(doc.to_dict())

print("Movies from 2017 starting with 'The':", movies_2017_the)

# TODO: Write a simple query operation to find all movies.
# TODO: Write a query operation to find all 'Action' movies.
# TODO: Write a query operation to find all movies released after 2016.

```

Here is your fully completed Firestore script with the three requested query implementations added at the bottom.

```python
from google.cloud import firestore
from google.cloud.firestore import FieldFilter

# Initialize Firestore client
db = firestore.Client()

# Reference to the Movies collection
movies_ref = db.collection('Movies')

# Clear any existing data in the collection
docs = movies_ref.stream()
for doc in docs:
    doc.reference.delete()

# Populate collection with movie data including the genre attribute
movies_ref.add({'year': 2016, 'title': 'The Big New Movie', 'genre': 'Comedy'})
movies_ref.add({'year': 2017, 'title': 'The Bigger, Newer Movie', 'genre': 'Action'})
movies_ref.add({'year': 2017, 'title': 'Yet Another Movie', 'genre': 'Action'})
movies_ref.add({'year': 2017, 'title': 'One More Movie', 'genre': 'Drama'})
movies_ref.add({'year': 2015, 'title': 'An Old Movie', 'genre': 'Romance'})
movies_ref.add({'year': 2018, 'title': 'Another New Movie', 'genre': 'Action'})

# Query for all movies released in 2018
query_2018 = movies_ref.where(filter=FieldFilter('year', '==', 2018))
results_2018 = query_2018.stream()

movies_2018 = []
for doc in results_2018:
    movies_2018.append(doc.to_dict())

print("Movies from 2018:", movies_2018)

# Query for all movies from 2017 with titles starting with "The"
query_2017_the = movies_ref.where(filter=FieldFilter('year', '==', 2017)).where(filter=FieldFilter('title', '>=', 'The')).where(filter=FieldFilter('title', '<', 'Thf'))
results_2017_the = query_2017_the.stream()

movies_2017_the = []
for doc in results_2017_the:
    movies_2017_the.append(doc.to_dict())

print("Movies from 2017 starting with 'The':", movies_2017_the)


# --- TODO FIXED: Write a simple query operation to find all movies ---
print("\n--- All Movies ---")
all_movies_stream = movies_ref.stream()
all_movies = [doc.to_dict() for doc in all_movies_stream]
print("All Movies in Collection:", all_movies)


# --- TODO FIXED: Write a query operation to find all 'Action' movies ---
print("\n--- Action Genre Movies ---")
query_action = movies_ref.where(filter=FieldFilter('genre', '==', 'Action'))
action_movies = [doc.to_dict() for doc in query_action.stream()]
print("Action Movies:", action_movies)


# --- TODO FIXED: Write a query operation to find all movies released after 2016 ---
print("\n--- Movies Released After 2016 ---")
query_post_2016 = movies_ref.where(filter=FieldFilter('year', '>', 2016))
post_2016_movies = [doc.to_dict() for doc in query_post_2016.stream()]
print("Movies released after 2016:", post_2016_movies)

```

---

## Query Breakdown and Architecture Insight

### 1. Retrieving All Documents

```python
all_movies_stream = movies_ref.stream()

```

* **The Mechanic:** In Firestore, you don't need a filter condition to pull everything. Calling `.stream()` directly on a collection reference triggers an index snapshot sequence that returns every document within that specific collection bucket.

### 2. Equality Searching (`==`)

```python
query_action = movies_ref.where(filter=FieldFilter('genre', '==', 'Action'))

```

* **The Mechanic:** This creates a clean key-value lookup against Firestore's automatically generated single-field index for `genre`. It isolates and transfers only the documents where the string completely matches.

### 3. Range-Based Inequality (`>`)

```python
query_post_2016 = movies_ref.where(filter=FieldFilter('year', '>', 2016))

```

* **The Mechanic:** Firestore indexes numerical values in sorted order. An inequality operator like `>` allows the backend scanner to jump right to the first index entry higher than `2016` and stream everything sequentially after it, ensuring the query scales perfectly even as millions of records are added.